<a href="https://colab.research.google.com/github/RageLog/bg-robust-rps/blob/main/train_on_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Background-Robust RPS — Colab research pipeline

This notebook runs the full experimental pipeline on Google Colab. It is fully modular:
use the end-to-end cell to run everything in one step, or the individual module cells to
run data generation, training, or Grad-CAM analysis independently.

## Initial setup (required)
Run the setup cell **once** at the start of a session. It clones the code from GitHub,
mounts Google Drive, and extracts the datasets.

In [ ]:
# @title 1. Environment setup (Drive, GitHub, datasets)
import os
import shutil
import zipfile
from google.colab import drive

# @markdown ### GitHub repository settings
GITHUB_REPO_URL = "https://github.com/RageLog/bg-robust-rps.git" # @param {type:"string"}
GITHUB_BRANCH = "main" # @param {type:"string"}

drive.mount('/content/drive')
DRIVE_DIR = "/content/drive/MyDrive/RPC_Colab"
DATA_ZIP = os.path.join(DRIVE_DIR, "datasets.zip")
PROJECT_DIR = "/content/rpc_project"
LOCAL_DIR = PROJECT_DIR

if not os.path.exists(DRIVE_DIR):
    print(f"Please create {DRIVE_DIR} in your Drive and upload datasets.zip into it.")
else:
    # Clone or update the repository
    if os.path.exists(PROJECT_DIR):
        print("Updating existing code (git pull)...")
        os.chdir(PROJECT_DIR)
        !git reset --hard
        !git pull origin {GITHUB_BRANCH}
    else:
        print("Cloning code from GitHub...")
        !git clone -b {GITHUB_BRANCH} {GITHUB_REPO_URL} {PROJECT_DIR}

    os.chdir(LOCAL_DIR)

    # Install dependencies
    if os.path.exists('requirements.txt'):
        print("Installing dependencies...")
        !pip install -q -r requirements.txt
        !pip install -q rembg[gpu]

    # Extract the dataset archive
    dataset_path = os.path.join(LOCAL_DIR, "datasets")
    raw_path = os.path.join(dataset_path, "raw")
    if not os.path.exists(raw_path) and os.path.exists(DATA_ZIP):
        print(f"Extracting {DATA_ZIP} to local storage...")
        os.makedirs(LOCAL_DIR, exist_ok=True)
        with zipfile.ZipFile(DATA_ZIP, 'r') as zip_ref:
            zip_ref.extractall(LOCAL_DIR)
        print("Dataset extracted.")
    else:
        print("Dataset already present, or the archive was not found in Drive.")

    print("\nSetup complete. The modules below can now be run independently.")

---
## Research modules (modular control panel)
Each cell below is independent. Run only the cell you need in order to test or debug a
single stage.

In [ ]:
# @title Module 1: synthetic data generation and splitting
import os
import shutil
os.chdir(LOCAL_DIR)

ABLATION_MODE = "FULL" # @param ["FULL", "INDOOR", "RANDBG", "REMBG_ONLY", "BASELINE"]
MODEL_NAME = "EfficientNetV2B0" # @param ["EfficientNetV2B0", "ResNet50", "MobileNetV3Small", "DenseNet121", "VGG16"]

print(f"Generating data | ablation: {ABLATION_MODE}")

# 1. Generate
!python src/generate_data.py --ablation {ABLATION_MODE} --model {MODEL_NAME}

# 2. Split
!python src/split_data.py --ablation {ABLATION_MODE}

# 3. Back up to Drive
synthetic_dir = os.path.join(LOCAL_DIR, "datasets", f"synthetic_{ABLATION_MODE.lower()}")
drive_zip_path = os.path.join(DRIVE_DIR, f"synthetic_{ABLATION_MODE.lower()}.zip")
if os.path.exists(synthetic_dir):
    print("Backing up dataset to Drive...")
    shutil.make_archive(synthetic_dir, 'zip', synthetic_dir)
    shutil.copy2(f"{synthetic_dir}.zip", drive_zip_path)
    print(f"Backed up: {drive_zip_path}")

In [ ]:
# @title Module 2: model training
import os
os.chdir(LOCAL_DIR)

ABLATION_MODE = "FULL" # @param ["FULL", "INDOOR", "RANDBG", "REMBG_ONLY", "BASELINE"]
MODEL_NAME = "EfficientNetV2B0" # @param ["EfficientNetV2B0", "ResNet50", "MobileNetV3Small", "DenseNet121", "VGG16"]
TUNE_STRATEGY = "standard" # @param ["standard", "progressive"]
HEAD_STRATEGY = "standard" # @param ["standard", "spatial_pooling", "attention"]
K_FOLD = 5 # @param {type:"slider", min:0, max:10, step:1}

print(f"Training | model: {MODEL_NAME} | ablation: {ABLATION_MODE} | k-fold: {K_FOLD}")

cmd = f"python src/train.py --ablation {ABLATION_MODE} --model {MODEL_NAME} --tune_strategy {TUNE_STRATEGY} --head_strategy {HEAD_STRATEGY}"
if K_FOLD > 1:
    cmd += f" --cv {K_FOLD}"

!{cmd}

# Back up models and reports to Drive
!cp -r models/ /content/drive/MyDrive/RPC_Colab/models/
!cp -r reports/ /content/drive/MyDrive/RPC_Colab/reports/
print("Models and reports copied to Drive.")

In [ ]:
# @title Module 3: explainability (Grad-CAM heatmaps)
import os
os.chdir(LOCAL_DIR)

ABLATION_MODE = "FULL" # @param ["FULL", "INDOOR", "RANDBG", "REMBG_ONLY", "BASELINE"]
MODEL_NAME = "EfficientNetV2B0" # @param ["EfficientNetV2B0", "ResNet50", "MobileNetV3Small", "DenseNet121", "VGG16"]
MODEL_SUFFIX = "best" # @param ["best", "fold_1_best", "fold_2_best"]

model_file = f"run1_{MODEL_NAME}_{ABLATION_MODE.lower()}_{MODEL_SUFFIX}.keras"
model_path = os.path.join(LOCAL_DIR, "models", model_file)

if not os.path.exists(model_path):
    print(f"Model not found: {model_path}")
    print("It may still be syncing from Drive into the models/ folder; please check.")
else:
    test_dir = os.path.join(LOCAL_DIR, "datasets", f"synthetic_{ABLATION_MODE.lower()}", "test")
    out_dir = os.path.join(LOCAL_DIR, "reports", MODEL_NAME, ABLATION_MODE.upper(), "gradcam")

    !python src/gradcam_vis.py --model_path {model_path} --test_dir {test_dir} --out_dir {out_dir}

    !cp -r reports/ /content/drive/MyDrive/RPC_Colab/reports/
    print("Grad-CAM reports copied to Drive.")

In [ ]:
# @title Module 4: cross-dataset evaluation and analyses
import os
os.chdir(LOCAL_DIR)

RUN_CROSS_DATASET = True # @param {type:"boolean"}
RUN_TEMPORAL_SMOOTHING = True # @param {type:"boolean"}
RUN_SEGMENTATION_EVAL = True # @param {type:"boolean"}

if RUN_CROSS_DATASET:
    print("\nRunning cross-dataset (zero-shot) evaluation...")
    !python src/evaluate_cross_dataset.py

if RUN_TEMPORAL_SMOOTHING:
    print("\nRunning temporal-smoothing optimisation...")
    !python src/optimize_temporal_smoothing.py

if RUN_SEGMENTATION_EVAL:
    print("\nRunning rembg segmentation-quality evaluation (pseudo-IoU / edge)...")
    !python src/evaluate_segmentation.py

!cp -r reports/ /content/drive/MyDrive/RPC_Colab/reports/
print("\nAnalysis reports copied to Drive.")

---
## End-to-end pipeline (all experiments)
The cell below runs the complete experimental grid. Once the modular cells above run
without error, it can be left to run unattended (for example, overnight).

In [ ]:
# @title End-to-end pipeline (run all)
import os
os.chdir(LOCAL_DIR)

RUN_MODE = "missing" # @param ["missing", "full"]
TUNE_STRATEGY = "both" # @param ["standard", "progressive", "both"]

print(f"Starting the full pipeline | mode: {RUN_MODE}")
!python run_all_experiments.py --run-mode {RUN_MODE} --tune_strategy {TUNE_STRATEGY}

---
## Recovery: regenerate reports from trained models
If training finished but the JSON reports were not produced (for example after a
disconnect), run this cell instead of repeating the k-fold training. It scans the
trained models, regenerates the reports from the validation set, and backs up all
weights and reports to Drive.

In [ ]:
# @title Regenerate missing reports and back up to Drive
import os
os.chdir(LOCAL_DIR)
print("Running the report-recovery script...")
!python regenerate_reports.py

---
## Q1 report generator
This cell scans the `.keras` weights and `.json` outputs on Drive and produces five
Excel-compatible CSV tables (ablation impact, per-class F1, generalisation gap, and
related summaries), then copies them to Drive.

In [ ]:
# @title Generate Q1 CSV tables and back up to Drive
import os
os.chdir(LOCAL_DIR)
print("Running the Q1 report and table generator...")

# This cell triggers the underlying Python script; the standalone notebook
# generate_ultimate_q1_reports_colab.ipynb can also be used directly.

!python src/generate_q1_academic_reports.py

!cp -r Q1_Reports/ /content/drive/MyDrive/RPC_Colab/reports/
print("\nQ1 CSV tables copied to Drive (reports/Q1_Reports/).")

In [ ]:
# @title Revision: external-none robustness probe (V2)
import os
os.chdir(LOCAL_DIR)

# Inference-only robustness check for the external "none" class. Reuses the existing
# .keras checkpoints (no training) and leaves the main pipeline untouched.
# Output: MyDrive/RPC_Colab/revize/global_cross_dataset_evaluation_v2.json
!python src/build_external_test_v2.py


In [ ]:
# @title Revision K1: multi-seed stability (representative subset)
import os, glob, zipfile, shutil
os.chdir(LOCAL_DIR)

# Re-trains the representative subset under additional random seeds to quantify seed
# stability (editor request). The main training code is unchanged; an additive wrapper
# patches config per seed. Datasets are staged from the Drive zips and split with
# split_data.py (seed 42, matching the original partition).
#
# Ablations whose dataset has no "none" class (e.g. NO_SYNTH) are skipped automatically.
# Jobs are ordered so all seeds of a cell finish together, so a partial run still yields
# complete cells. aggregate_multiseed.py flags any diverged fold; the run is resumable.
# PARALLEL=2 was validated as stable. Do not run the cleanup cell here: previous results
# on Drive must be preserved so the resume logic can skip completed cells.

MULTISEED_SEEDS     = "42,123,2024"  # @param {type:"string"}
MULTISEED_MODELS    = "DenseNet121,EfficientNetV2B0,ResNet50,MobileNetV3Small,VGG16"  # @param {type:"string"}
MULTISEED_ABLATIONS = "FULL,INDOOR,RANDBG,REMBG_ONLY,NO_ALPHA,NO_SHIFT,RATIO_1X,GAN,STYLE_TRANSFER"  # @param {type:"string"}
MULTISEED_PARALLEL  = "2"  # @param {type:"string"}

os.environ["MULTISEED_SEEDS"]     = MULTISEED_SEEDS
os.environ["MULTISEED_MODELS"]    = MULTISEED_MODELS
os.environ["MULTISEED_ABLATIONS"] = MULTISEED_ABLATIONS
os.environ["MULTISEED_PARALLEL"]  = MULTISEED_PARALLEL

# Automatic dataset staging: Drive zip -> datasets/synthetic_<mode>.
DRIVE = "/content/drive/MyDrive/RPC_Colab"
CLASSES = {"rock", "paper", "scissors", "none"}

def _imgs(d):
    return glob.glob(os.path.join(d, "train", "*", "*.jpg")) + glob.glob(os.path.join(d, "train", "*", "*.png"))

def _stage(ab):
    dest = os.path.join(LOCAL_DIR, "datasets", f"synthetic_{ab}")
    if _imgs(dest):
        print(f"  synthetic_{ab}: ready ({len(_imgs(dest))} train images)"); return True
    has_classes = os.path.isdir(dest) and (set(os.listdir(dest)) & CLASSES)
    if not has_classes:
        z = os.path.join(DRIVE, f"synthetic_{ab}.zip")
        if not os.path.exists(z):
            print(f"  synthetic_{ab}: zip not found at {z}, skipping"); return False
        if os.path.isdir(dest): shutil.rmtree(dest)
        os.makedirs(dest, exist_ok=True)
        print(f"  synthetic_{ab}: extracting ({os.path.getsize(z)//1_000_000} MB)...")
        with zipfile.ZipFile(z) as zf: zf.extractall(dest)
        subs = [d for d in os.listdir(dest) if os.path.isdir(os.path.join(dest, d))]
        if not os.path.isdir(os.path.join(dest, "train")) and len(subs) == 1 and subs[0] not in CLASSES:
            inner = os.path.join(dest, subs[0])
            for it in os.listdir(inner): shutil.move(os.path.join(inner, it), os.path.join(dest, it))
            shutil.rmtree(inner)
    if not os.path.isdir(os.path.join(dest, "train")) and (set(os.listdir(dest)) & CLASSES):
        print(f"  synthetic_{ab}: splitting raw class folders (split_data.py, seed 42)")
        os.system(f"python src/split_data.py --ablation {ab}")
    n = len(_imgs(dest))
    none_n = len(glob.glob(os.path.join(dest, "train", "none", "*")))
    print(f"  synthetic_{ab}: train={n}, none={none_n}{' (3-class, will be skipped)' if none_n == 0 else ''}")
    return n > 0

print("=== Dataset staging ===")
_abs = [a.strip().lower() for a in MULTISEED_ABLATIONS.split(",") if a.strip()]
_ok = any([_stage(ab) for ab in _abs])
print("  (seed-42 baseline is read directly from Drive/reports during aggregation)")

if not _ok:
    raise SystemExit("No usable dataset found; training not started.")
print(f"Datasets ready. Starting multi-seed run (PARALLEL={MULTISEED_PARALLEL}).\n")

!python src/verify_multiseed_splits.py && \
 python src/train_multiseed.py && \
 python src/aggregate_multiseed.py && \
 python paper/scripts/make_multiseed_table.py

!cp -r reports_multiseed/ /content/drive/MyDrive/RPC_Colab/revize/ 2>/dev/null
print("\nDone. Summary: reports_multiseed/multiseed_summary.json | Table: paper/tables/T17_multiseed_stability.tex")
